# Heart Disease Prediction ❤️

## 1. Datasets & DataLoaders

In [83]:
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
import pandas as pd
from torch.utils.data.dataset import Dataset
import torch.utils.data
from torch.utils.data import DataLoader
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

class HeartData(Dataset):
    def __init__(self, file_path):
        raw_data = pd.read_csv(file_path)

        x = raw_data.values[:, :-1].astype("float32")
        y = raw_data.values[:, -1].astype("int64")

        from sklearn.preprocessing import MinMaxScaler
        scaler = MinMaxScaler()
        x = scaler.fit_transform(x).astype("float32")

        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]



def get_data(batch_s):
    heart_data = HeartData("heart.csv")

    training_data, test_data = torch.utils.data.random_split(
        heart_data,
        [int(len(heart_data) * 0.9),
         len(heart_data) - int(len(heart_data) * 0.9)],
    )
    train_dataloader = DataLoader(training_data, batch_size=batch_s, shuffle=True)
    test_dataloader = DataLoader(test_data, batch_size=batch_s, shuffle=True)
    return train_dataloader, test_dataloader


## 2. Build Model

In [105]:
class HeartNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(13, 20),
            nn.ReLU(),
            nn.Linear(20, 20),
            nn.ReLU(),
            nn.Linear(20, 2)
        )

    def forward(self, x):
        return self.network(x)

model = HeartNetwork()

## 3. Optimization

In [124]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        pred = model(X)
        loss = loss_fn(pred, y)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 1 == 0:
            loss, current = loss.item(), batch * dataloader.batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
            
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")


## 4. Main

In [130]:
model = HeartNetwork()

In [131]:
learning_rate = 0.03
epochs = 5
batch_size = 100
train_dataloader, test_dataloader = get_data(batch_size)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 0.722302  [  100/  922]
loss: 0.669572  [  200/  922]
loss: 0.645198  [  300/  922]
loss: 0.616990  [  400/  922]
loss: 0.542860  [  500/  922]
loss: 0.523887  [  600/  922]
loss: 0.490823  [  700/  922]
loss: 0.421104  [  800/  922]
loss: 0.419475  [  900/  922]
loss: 0.178303  [  922/  922]
Test Error: 
 Accuracy: 82.5%, Avg loss: 0.628918 

Epoch 2
-------------------------------
loss: 0.340569  [  100/  922]
loss: 0.544086  [  200/  922]
loss: 0.324642  [  300/  922]
loss: 0.378620  [  400/  922]
loss: 0.506164  [  500/  922]
loss: 0.475604  [  600/  922]
loss: 0.444506  [  700/  922]
loss: 0.512468  [  800/  922]
loss: 0.470969  [  900/  922]
loss: 0.344588  [  922/  922]
Test Error: 
 Accuracy: 75.7%, Avg loss: 0.452155 

Epoch 3
-------------------------------
loss: 0.359339  [  100/  922]
loss: 0.357883  [  200/  922]
loss: 0.329562  [  300/  922]
loss: 0.391286  [  400/  922]
loss: 0.471449  [  500/  922]
loss: 0.500151  [  600/  9

## 5. Save and load the model

In [137]:
import torchvision.models as models
device = "cpu"
torch.save(model.state_dict(), 'heart_model.pth')

In [138]:
model.load_state_dict(torch.load('heart_model.pth', map_location=device))
model.eval()
print("Model loaded")

Model loaded
